In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from gensim.models import KeyedVectors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [7]:
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/TA DIAZ/datasetdiaz.csv', usecols=['HS', 'Tweet'])
df = df.dropna(subset=['Tweet', 'HS'])  # Hilangkan NaN

X = np.array(df['Tweet'])
y = np.array(df['HS'])

In [16]:
# Load Word2Vec model
model_word2vec = KeyedVectors.load_word2vec_format('/content/drive/MyDrive/TA DIAZ/ws_model100.bin', binary=True)

In [17]:
# Fungsi untuk mengubah teks menjadi vektor Word2Vec
def get_w2v_features(w2v_model, sentence_group):
    feature_vec = np.zeros(w2v_model.vector_size, dtype="float32")  # Inisialisasi vektor nol
    num_words = 0
    for word in sentence_group.split():
        if word in w2v_model:
            feature_vec += w2v_model[word]  # Tambahkan vektor kata
            num_words += 1
    if num_words > 0:
        feature_vec /= num_words  # Rata-rata jika ada kata yang valid
    return feature_vec

In [18]:
# Konversi teks ke fitur Word2Vec
X_w2v = np.array([get_w2v_features(model_word2vec, text) for text in X])


In [19]:
# Standardisasi fitur Word2Vec
scaler = StandardScaler()
X_w2v_scaled = scaler.fit_transform(X_w2v)

In [20]:
# Fungsi untuk split train-test
def split_train_test(X, y, test_size):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=25)
    num_classes = np.max(y) + 1
    y_train = to_categorical(y_train, num_classes)
    y_test = to_categorical(y_test, num_classes)
    return X_train, X_test, y_train, y_test

In [21]:
# Fungsi CNN Model
def cnn_model(X, y, test_size):
    # Split dataset
    X_train, X_test, y_train, y_test = split_train_test(X, y, test_size)

    # Reshape data agar sesuai input CNN
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Define CNN Model
    model = Sequential([
        Conv1D(32, 5, activation='relu', input_shape=(X_train.shape[1], 1), padding='same'),
        MaxPooling1D(5, padding='same'),
        Dropout(0.3),
        Flatten(),
        Dense(32, activation='relu'),
        Dense(2, activation='sigmoid')  # Sesuai dengan kode asli Anda
    ])
    # Compile model
    optimizer = Adam(learning_rate=0.0001)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    # Early Stopping untuk mencegah overfitting
    early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

    # Train model
    model.fit(X_train, y_train, epochs=35, batch_size=16, verbose=1, validation_data=(X_test, y_test), callbacks=[early_stopping])

    # Evaluasi model
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

    # Prediksi pada data uji
    y_pred = model.predict(X_test)

    # Hitung metrik evaluasi
    accuracy = accuracy_score(y_test.argmax(axis=1), y_pred.argmax(axis=1))
    precision = precision_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    recall = recall_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    f1 = f1_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    class_repot = classification_report(y_test.argmax(axis=1), y_pred.argmax(axis=1))

    return accuracy, precision, recall, f1, class_repot

In [22]:
# Menjalankan eksperimen 5 kali seperti di kode asli
num_experiments = 5
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

for i in range(num_experiments):
    print(f"Experiment {i+1}")

    # Jalankan CNN dengan Word2Vec
    accuracy, precision, recall, f1, class_repot = cnn_model(X_w2v_scaled, y, 0.2)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    # Print hasil tiap eksperimen
    print(f"Evaluation Metrics - Experiment {i+1}:")
    print(f"Accuracy Score: {accuracy:.4f}")
    print(f"Precision Score: {precision:.4f}")
    print(f"Recall Score: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print()
    print("Classification Report:")
    print(class_repot)
    print('-------------------------------------------------------------------')
    print()


Experiment 1
Epoch 1/35


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5566 - loss: 0.6858 - val_accuracy: 0.6140 - val_loss: 0.6620
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6200 - loss: 0.6562 - val_accuracy: 0.6619 - val_loss: 0.6403
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6430 - loss: 0.6387 - val_accuracy: 0.6514 - val_loss: 0.6248
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6562 - loss: 0.6290 - val_accuracy: 0.6584 - val_loss: 0.6178
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6615 - loss: 0.6210 - val_accuracy: 0.6675 - val_loss: 0.6115
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6679 - loss: 0.6131 - val_accuracy: 0.6867 - val_loss: 0.6025
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6711 - loss: 0.6046 - val_accuracy: 0.6915 - val_loss: 0.5977
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6782 - loss: 0.6048 - val_accuracy: 0.6902 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5514 - loss: 0.6886 - val_accuracy: 0.5796 - val_loss: 0.6723
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5879 - loss: 0.6717 - val_accuracy: 0.6223 - val_loss: 0.6542
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6195 - loss: 0.6554 - val_accuracy: 0.6510 - val_loss: 0.6368
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6443 - loss: 0.6397 - val_accuracy: 0.6701 - val_loss: 0.6237
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6486 - loss: 0.6309 - val_accuracy: 0.6575 - val_loss: 0.6175
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6642 - loss: 0.6204 - val_accuracy: 0.6849 - val_loss: 0.6090
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6676 - loss: 0.6137 - val_accuracy: 0.6810 - val_loss: 0.6055
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6851 - loss: 0.6067 - val_accuracy: 0.6836 - val_

...

575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5685 - loss: 0.6851 - val_accuracy: 0.5944 - val_loss: 0.6638
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.6013 - loss: 0.6644 - val_accuracy: 0.6245 - val_loss: 0.6447
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6319 - loss: 0.6468 - val_accuracy: 0.6414 - val_loss: 0.6316
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6403 - loss: 0.6361 - val_accuracy: 0.6797 - val_loss: 0.6185
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6699 - loss: 0.6172 - val_accuracy: 0.6849 - val_loss: 0.6115
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6678 - loss: 0.6183 - val_accuracy: 0.6893 - val_loss: 0.6049
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6644 - loss: 0.6165 - val_accuracy: 0.6923 - val_loss: 0.5995
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6789 - loss: 0.6115 - val_accuracy: 0.6862 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5497 - loss: 0.6904 - val_accuracy: 0.5962 - val_loss: 0.6653
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5995 - loss: 0.6631 - val_accuracy: 0.6366 - val_loss: 0.6378
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6425 - loss: 0.6417 - val_accuracy: 0.6471 - val_loss: 0.6275
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6534 - loss: 0.6292 - val_accuracy: 0.6710 - val_loss: 0.6122
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6574 - loss: 0.6257 - val_accuracy: 0.6910 - val_loss: 0.6050
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6700 - loss: 0.6146 - val_accuracy: 0.6928 - val_loss: 0.5965
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6784 - loss: 0.6050 - val_accuracy: 0.6954 - val_loss: 0.5929
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6762 - loss: 0.6018 - val_accuracy: 0.7032 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5681 - loss: 0.6821 - val_accuracy: 0.6240 - val_loss: 0.6512
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6226 - loss: 0.6541 - val_accuracy: 0.6445 - val_loss: 0.6291
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6518 - loss: 0.6312 - val_accuracy: 0.6728 - val_loss: 0.6120
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6506 - loss: 0.6280 - val_accuracy: 0.6897 - val_loss: 0.6017
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6602 - loss: 0.6148 - val_accuracy: 0.6923 - val_loss: 0.5956
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6834 - loss: 0.6027 - val_accuracy: 0.7002 - val_loss: 0.5909
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6770 - loss: 0.6017 - val_accuracy: 0.6932 - val_loss: 0.5878
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6815 - loss: 0.6004 - val_accuracy: 0.6997 - val_

In [23]:
# Rata-rata hasil dari 5 eksperimen
avg_accuracy = np.mean(accuracy_scores)
avg_precision = np.mean(precision_scores)
avg_recall = np.mean(recall_scores)
avg_f1 = np.mean(f1_scores)

print("\nAverage Evaluation Metrics:")
print(f"Average Accuracy Score: {avg_accuracy:.4f}")
print(f"Average Precision Score: {avg_precision:.4f}")
print(f"Average Recall Score: {avg_recall:.4f}")
print(f"Average F1 Score: {avg_f1:.4f}")



Average Evaluation Metrics:
Average Accuracy Score: 0.7288
Average Precision Score: 0.7317
Average Recall Score: 0.7181
Average F1 Score: 0.7196
